In [7]:
# Import required libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.types import DoubleType

# Create Spark Session
spark = SparkSession.builder \
    .appName("Week 6 Advanced PySpark Assignment") \
    .getOrCreate()

# Q1: Explain the roles of the Driver, Cluster Manager, and Executor in a Spark application.
# Answer
1] Driver
The Driver is the main program of a Spark application. It creates the SparkSession, converts the application into tasks, and coordinates the execution of the entire job.

2] Cluster Manager
The Cluster Manager is responsible for allocating resources such as CPU and memory to the Spark application. It manages all worker nodes available in the cluster.

3] Executor
Executors run on worker nodes. They execute the tasks assigned by the Driver, process the data, and send the results back to the Driver.

4] Insight
The Driver controls the application, the Cluster Manager manages the resources, and Executors perform the actual data processing.

# Q2: How does Spark's Lazy Evaluation strategy improve performance when chain-processing large datasets?
# Answer
Spark uses Lazy Evaluation, which means transformations are not executed immediately.
Instead, Spark records all transformations in a Directed Acyclic Graph (DAG). The execution starts only when an Action such as show(), count(), or collect() is called.
This helps Spark optimize the execution plan by combining multiple transformations into a single job.

-Advantages
Reduces unnecessary computation.
Optimizes execution before running.
Improves performance.
Minimizes resource usage.

-Insight
Lazy Evaluation makes Spark faster by executing only the required operations after creating an optimized execution plan.

In [48]:
# Q3: Write a Spark command to read a CSV file located at "data/source.csv",
# ensuring the first row is treated as a header and inferSchema is enabled.

# Answer

df = spark.read.csv(
    r"C:\Week-6_Advanced_PySpark_Assignment\data\Sample - Superstore.csv",
    header=True,
    inferSchema=True
)

df.show(5)

df.printSchema()

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

# Q4: What is the difference between CSV and Parquet in terms of storage (row-based vs. columnar) and why does it matter for performance?
# Answer

-CSV is a row-based file format. It stores data row by row and usually requires more storage space. While reading a CSV file, Spark scans the entire dataset even if only a few columns are required.

-Parquet
Parquet is a columnar file format. It stores data column by column, supports compression, and reads only the required columns. This reduces disk I/O and improves query performance.

-Difference
CSV is easier to read and share but slower for large-scale analytics.
Parquet provides better compression, faster query execution, and lower storage usage.

-Insight
For Data Engineering projects, Parquet is generally preferred because it improves storage efficiency and processing performance compared to CSV.

In [49]:
# Q5: Given a DataFrame df, write a query to select the columns
# Product Name and Sales where the Category is 'Technology'.

# Answer

from pyspark.sql.functions import col

df.select("Product Name", "Sales") \
  .filter(col("Category") == "Technology") \
  .show()

+--------------------+--------+
|        Product Name|   Sales|
+--------------------+--------+
|Mitel 5320 IP Pho...| 907.152|
|Konftel 250 Confe...| 911.424|
|Cisco SPA 501G IP...|  213.48|
|Imation�8GB Mini ...|   90.57|
|         GE 30524EE4|1097.544|
|Plantronics HL10 ...| 371.168|
|  Panasonic Kx-TS550| 147.168|
|Verbatim 25 GB 6x...|   45.98|
|Imation�8gb Micro...|      45|
|LF Elite 3D Dazzl...|    21.8|
|AT&T CL83451 4-Ha...| 1029.95|
|Imation�8gb Micro...|      30|
|Verbatim 25 GB 6x...|   13.98|
|netTALK DUO VoIP ...| 167.968|
|Logitech�LS21 Spe...|   19.99|
|  Panasonic Kx-TS550|  73.584|
|SanDisk Ultra 64 ...|  95.976|
|Logitech K350 2.4...| 238.896|
|Memorex Mini Trav...|  74.112|
|Speck Products Ca...|  27.992|
+--------------------+--------+
only showing top 20 rows


In [50]:
# Q6: Rename the column Customer Name to Customer_Name
# and cast the Sales column from String to Double.

# Answer

from pyspark.sql.functions import col

df = df.withColumnRenamed(
    "Customer Name",
    "Customer_Name"
)

df = df.withColumn(
    "Sales",
    col("Sales").cast("double")
)

df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)



# Q7: How does Spark use the Lineage Graph (DAG) to provide fault tolerance if a worker node fails?
# Answer:
-park maintains a Lineage Graph (DAG) that records all transformations applied to the data.

-If an Executor or Worker node fails, Spark does not reload the entire dataset.
-Instead, it uses the Lineage Graph to recompute only the missing partitions from the original data.
-This makes Spark fault tolerant and avoids unnecessary recomputation.

-Insight
The Lineage Graph allows Spark to recover lost data efficiently without storing multiple copies of the dataset.

In [54]:
#re-read the datase to run the Q8
df = spark.read.csv(
    r"C:\Week-6_Advanced_PySpark_Assignment\data\Sample - Superstore.csv",
    header=True,
    inferSchema=True,
    multiLine=True,
    escape='"'
)

In [55]:
# Q8: Write a query to filter a DataFrame
# where Ship Mode is 'Second Class'
# and Sales is greater than 1000.

# Answer

from pyspark.sql.functions import col

# Convert Sales column to Double
df = df.withColumn(
    "Sales",
    col("Sales").cast("double")
)

# Filter the required rows
df.filter(
    (col("Ship Mode") == "Second Class") &
    (col("Sales") > 1000)
).show()

+------+--------------+----------+----------+------------+-----------+-----------------+-----------+-------------+---------------+----------+-----------+-------+---------------+---------------+------------+--------------------+---------+--------+--------+---------+
|Row ID|      Order ID|Order Date| Ship Date|   Ship Mode|Customer ID|    Customer Name|    Segment|      Country|           City|     State|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name|    Sales|Quantity|Discount|   Profit|
+------+--------------+----------+----------+------------+-----------+-----------------+-----------+-------------+---------------+----------+-----------+-------+---------------+---------------+------------+--------------------+---------+--------+--------+---------+
|   245|CA-2014-131926|  6/1/2014|  6/6/2014|Second Class|   DW-13480|    Dianna Wilson|Home Office|United States|      Lakeville| Minnesota|      55044|Central|FUR-CH-10004063|      Furniture|      Cha

# Q9: Explain Predicate Pushdown in Parquet.
# Answer:
Predicate Pushdown is an optimization technique used by Spark when reading Parquet files.
Instead of reading the entire dataset into memory, Spark pushes the filter condition down to the Parquet file.

The Parquet file reads only the rows that satisfy the filter condition and skips unnecessary data.
This reduces the amount of data loaded into memory, improves query performance, and decreases disk I/O.
For example, if we filter records where Region = "West", Spark reads only the required rows instead of scanning the complete dataset.

-Insight
Predicate Pushdown makes Spark applications faster by reading only the required data from Parquet files, reducing memory usage and improving performance.

# Example of Predicate Pushdown

from pyspark.sql.functions import col

parquet_df = spark.read.parquet("path/to/parquet_file")

parquet_df.filter(
    col("Region") == "West"
).show()

 # Why this code?
The original question is about Predicate Pushdown.
Since our dataset contains the Region column, it is used for filtering.
When using a Parquet file, Spark reads only the matching rows instead of scanning the entire dataset, which improves performance.

In [56]:
# Q10: Write a code snippet to add a new column final_price
# which is the Sales multiplied by 1.18 (18% tax).

# Answer

from pyspark.sql.functions import col

# Add a new column final_price
df = df.withColumn(
    "final_price",
    col("Sales") * 1.18
)

# Display required columns
df.select(
    "Product Name",
    "Sales",
    "final_price"
).show(5)

+--------------------+--------+------------------+
|        Product Name|   Sales|       final_price|
+--------------------+--------+------------------+
|Bush Somerset Col...|  261.96|309.11279999999994|
|Hon Deluxe Fabric...|  731.94|          863.6892|
|Self-Adhesive Add...|   14.62|           17.2516|
|Bretford CR4500 S...|957.5775|        1129.94145|
|Eldon Fold 'N Rol...|  22.368|26.394239999999996|
+--------------------+--------+------------------+
only showing top 5 rows


In [36]:
#Q11: What is the difference between Transformations and Actions? Provide two examples of each.
#Answer:
#Transformations
#Transformations create a new DataFrame or RDD but do not execute immediately.
#Spark records these operations in a Lineage Graph (DAG) and executes them only when an Action is called (Lazy Evaluation).
#Examples of Transformations:

#select()
#filter()

# Transformation Examples

# Select specific columns
df.select("Customer Name", "Sales")

# Filter rows where Sales > 1000
df.filter(col("Sales") > 1000)


# Action Examples

# Display first 5 rows
df.show(5)

# Count total rows
df.count()

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+------------------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|       final_price|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+------------------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FU

9994

In [ ]:
# Q12: Write the Spark command to load a Parquet file from "path/to/input",
# filter out any rows where user_id is null,
# and save the result as a CSV at "path/to/output".

# Answer

from pyspark.sql.functions import col

# Read the Parquet file
parquet_df = spark.read.parquet("path/to/input")

# Filter rows where Customer ID is not null
# (Customer ID is used because our Sample Superstore dataset
# does not contain a user_id column.)
filtered_df = parquet_df.filter(
    col("Customer ID").isNotNull()
)

# Save the filtered data as CSV
filtered_df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("path/to/output")

# Display first 5 rows
filtered_df.show(5)

# Q13: In Spark Architecture, what is the difference between Client Mode and Cluster Mode
# Answer
-Client Mode
In Client Mode, the Driver program runs on the local machine where the Spark application is submitted. The Executors run on the worker nodes in the cluster.

-Cluster Mode
In Cluster Mode, both the Driver and Executors run inside the cluster. The Cluster Manager is responsible for launching and managing the Driver.

-Difference
Client Mode is mainly used for development and testing because the Driver runs on the user's machine.
Cluster Mode is preferred in production because the Driver runs inside the cluster, making the application more reliable even if the client machine disconnects.

-Insight
Client Mode is suitable for development, while Cluster Mode is better for production environments because the entire application runs within the cluster.

In [57]:
# Q14: Write a query to filter a dataset for rows
# where Region is 'North'
# OR Segment is 'Consumer'.

# Answer

from pyspark.sql.functions import col

df.filter(
    (col("Region") == "North") |
    (col("Segment") == "Consumer")
).show()

+------+--------------+----------+----------+--------------+-----------+------------------+--------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+------------------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|     Customer Name| Segment|      Country|           City|         State|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|       final_price|
+------+--------------+----------+----------+--------------+-----------+------------------+--------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+------------------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|       Claire Gute|Consumer|United States|      Henderson|      Ken

# Q15: When exploring a dataset, why is it safer to use .show(5) instead of .collect() on a multi-terabyte dataset?
# Answer

The show(5) function displays only the first five rows of the dataset without loading the entire dataset into the Driver's memory.
The collect() function retrieves all records from the distributed cluster and stores them in the Driver's memory.
For very large datasets, using collect() can consume a large amount of memory and may cause the application to become slow or even fail due to an OutOfMemoryError.
Therefore, show(5) is the safer and recommended option when exploring large datasets.

-Insight
Always use show() to preview large datasets. Use collect() only when the dataset is small enough to fit into the Driver's memory.